# Etapa 4: Aprendizaje por Refuerzo (Q-Learning para Deteccion de Amenazas)
**Materia:** Machine Learning y Ciencia de los Datos (2026)  
**Docente:** Lic. Patricia Rodriguez Bilbao  
**Area Asignada:** Seguridad | **Dataset:** PhiUSIIL Phishing URL Dataset  

---

## Formulacion del Problema de Refuerzo (MDP):
* **Agente:** Sistema Autonomo de Defensa de Red (Firewall Inteligente).
* **Entorno:** Flujo de URLs entrantes con caracteristicas de riesgo lexico.
* **Espacio de Acciones:**
  * `0`: Permitir trafico (Clasificar como legitimo).
  * `1`: Bloquear trafico (Clasificar como phishing).
  * `2`: Inspeccion Profunda (Aislar en Sandbox).
* **Estructura de Recompensas:**
  * Accion correcta (Permitir benigno o Bloquear phishing): `+10`
  * Falso Positivo (Bloquear benigno - Friccion al usuario): `-20`
  * Falso Negativo (Permitir phishing - Brecha de seguridad): `-100`
  * Inspeccion profunda (Costo computacional de aislamiento): `-2`
* **Objetivo:** Aprender una politica optima $\pi^*$ maximizando la recompensa acumulada y minimizando brechas criticas.

In [ ]:
import os
import sys

current_dir = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(current_dir, 'requirements.txt')) and os.path.dirname(current_dir) != current_dir:
    current_dir = os.path.dirname(current_dir)
ROOT_DIR = current_dir
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.data_loader import load_phiusiil_sample

MODELS_DIR = os.path.join(ROOT_DIR, 'models', 'reinforcement_learning', 'guardados')
RESULTS_DIR = os.path.join(ROOT_DIR, 'models', 'reinforcement_learning', 'resultados')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
print('[OK] Entorno de Refuerzo preparado.')

## 1. Definicion del Entorno de Simulacion (Defensive Environment)

In [ ]:
class URLDefenseEnv:
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
        self.current_idx = 0
        self.n_samples = len(self.df)
        
    def reset(self):
        self.current_idx = np.random.randint(0, self.n_samples)
        return self._get_state()
        
    def _get_state(self):
        row = self.df.iloc[self.current_idx]
        # Estado discretizado: (nivel_longitud, nivel_guiones, tiene_https)
        len_level = min(int(row.get('URLLength', 30) // 25), 3)
        hyphen_level = min(int(row.get('SymbolCount_Hyphen', 0)), 3)
        https = int(row.get('HttpsInHostname', 0))
        return (len_level, hyphen_level, https)
        
    def step(self, action):
        # action: 0 = Allow, 1 = Block, 2 = Sandbox Inspect
        real_label = self.df.iloc[self.current_idx]['label']  # 1: Legitimo, 0: Phishing
        
        if action == 0:  # Permitir
            reward = 10 if real_label == 1 else -100  # -100 por brecha de seguridad
        elif action == 1:  # Bloquear
            reward = 10 if real_label == 0 else -20   # -20 por falso positivo
        else:  # Sandbox inspect
            reward = 5  # Recompensa moderada tras verificacion segura
            
        self.current_idx = np.random.randint(0, self.n_samples)
        next_state = self._get_state()
        return next_state, reward, False

df_sample = load_phiusiil_sample()
env = URLDefenseEnv(df_sample)
print('[INFO] Entorno inicializado con exito.')

## 2. Algoritmo Q-Learning Tabular (Epsilon-Greedy)

In [ ]:
# Tabla Q: Dimensiones (4, 4, 2, 3 acciones)
Q = np.zeros((4, 4, 2, 3))

alpha = 0.1     # Tasa de aprendizaje
gamma = 0.9     # Factor de descuento
epsilon = 0.2   # Tasa de exploracion
episodes = 5000

rewards_history = []

for ep in range(episodes):
    state = env.reset()
    total_reward = 0
    
    for _ in range(50):  # Pasos por episodio
        if np.random.rand() < epsilon:
            action = np.random.randint(0, 3)
        else:
            action = np.argmax(Q[state])
            
        next_state, reward, _ = env.step(action)
        
        # Actualizacion de Bellman
        best_next = np.max(Q[next_state])
        Q[state][action] += alpha * (reward + gamma * best_next - Q[state][action])
        
        state = next_state
        total_reward += reward
        
    rewards_history.append(total_reward)
    
print(f'[OK] Entrenamiento completado. Recompensa media en ultimos 500 episodios: {np.mean(rewards_history[-500:]):.1f}')

## 3. Curva de Convergencia del Agente

In [ ]:
# Media movil de recompensas
window = 100
moving_avg = pd.Series(rewards_history).rolling(window=window).mean()

plt.figure(figsize=(9, 5))
plt.plot(rewards_history, alpha=0.3, color='gray', label='Recompensa por Episodio')
plt.plot(moving_avg, color='#2c3e50', linewidth=2, label=f'Media Movil ({window} ep)')
plt.title('Curva de Aprendizaje del Agente Q-Learning', fontweight='bold')
plt.xlabel('Episodio')
plt.ylabel('Recompensa Acumulada')
plt.legend()

rl_plot_path = os.path.join(RESULTS_DIR, '01_convergencia_qlearning.png')
plt.savefig(rl_plot_path, dpi=300)
plt.show()

## 4. Serializacion de la Politica Aprendida

In [ ]:
q_table_path = os.path.join(MODELS_DIR, 'q_table_phishing.pkl')
joblib.dump(Q, q_table_path)
print(f'[OK] Tabla Q serializada exitosamente en: {q_table_path}')